# Activation patching: where do failed transfers lose the update?

Proposal section III.C. The cases are the `transfer_fail` rows from the lens run (`patch_cases.csv`): the trained model gives the **new** answer in English but not in French, Chinese or Finnish. For each case and each layer *l*, we rerun the target-language prompt with its hidden state at layer *l* (at one token position) replaced by the English run's state at the same layer:

- **Δ_l = D_patched,l − D_baseline**, in the same units as `D` in `lens.csv` (log p(new) − log p(old), first token).
- **Control:** the same patch from a *different* fact's English run. The gap between the two (`delta_specific`) is the part that's about this fact, not just "English state in a French run".
- **Positions:**
  - `last`: the final prompt token, where the answer is predicted.
  - `subject`: the last token of the entity name in the question, where factual recall usually happens.

**How to read it:**
- **Recovery from mid-layer patches:** the target-language run can use the English representation from there on, so its own version was missing or wrong. That points to an *encoding/retrieval* failure.
- **Recovery only from late patches at the last position:** the update was lost in the final layers (*decoding*).
- **Nothing beats the control:** this position and layer don't carry the update.

**Setup:** Rohan's round-2 adapter with the three-example prompts (`results/rohan_fewshot`), which has the most failure cases (44). GPU runtime (T4 is fine), about 5 minutes. The checks in step 1 run first, on this model.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/64610"
%cd {PROJECT_DIR}
import sys
sys.path.insert(0, PROJECT_DIR)

In [ ]:
!pip -q install "transformers==4.57.6" "peft==0.18.0"
!pip -q uninstall -y torchao

In [ ]:
BASE      = "Qwen/Qwen3-1.7B-Base"
FACTS     = "data/rohan_facts_fewshot.jsonl"
ADAPTER   = "adapters/rohan/round2"            # the trained model the failures come from
RUN_DIR   = "results/rohan_fewshot"            # has status.csv and patch_cases.csv from run_rohan.ipynb
POSITIONS = ("last", "subject")
CHUNK     = None     # layers per batch; set e.g. 8 if the GPU runs out of memory

## 1. Load the trained model and run the patching checks
`run_checks` (from `tests/check_patching.py`) verifies the patching on this exact model:
- a self-patch changes nothing;
- the last-layer patch reproduces the English output;
- the embedding-layer patch matches a manual run;
- earlier layers and positions stay untouched;
- batched and single runs agree;
- hooks are cleaned up;
- the baseline `D` matches the lens pipeline.

If any check fails, stop there.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from xling.modeling import load_model, attach_adapter
from xling.schema import load_facts
from tests.check_patching import run_checks

model, tok = load_model(BASE)
model = attach_adapter(model, ADAPTER)
facts = load_facts(FACTS)
run_checks(model, tok, facts)

## 2. Patch every failure case at every layer

In [ ]:
from xling.patching import run_patching, summarize_patching, peak_layers

cases = pd.read_csv(f"{RUN_DIR}/patch_cases.csv")
cases = cases[cases.model == "round2"] if "model" in cases else cases
print(len(cases), "cases:", cases.lang.value_counts().to_dict())

patch = run_patching(model, tok, facts, cases, model_tag="round2", positions=POSITIONS, chunk=CHUNK)
patch.to_csv(f"{RUN_DIR}/patching.csv", index=False)
print("skipped:", patch.attrs["skipped"])
print(f"wrote {RUN_DIR}/patching.csv ({len(patch)} rows)")

## 3. Results
Colored line: patch from the same fact's English run. Gray: from a different fact (control). Positive Δ moves the target run toward the new answer.

In [ ]:
from xling import plots
import os
os.makedirs(f"{RUN_DIR}/figures", exist_ok=True)
for pos in POSITIONS:
    fig = plots.plot_patching(patch, pos)
    fig.savefig(f"{RUN_DIR}/figures/patching_{pos}.png", dpi=160, bbox_inches="tight")
    plt.show()

print("Layer where the fact-specific effect peaks, per position and language:")
display(peak_layers(patch).round(2))

### Does the patch keep the answer in the target language?
Median probability on the target-language answers' first tokens vs on the English answers' first tokens, after patching. If English takes over at some layers, the gain there comes from copying English output, not from the target run using the English representation.

In [ ]:
s = summarize_patching(patch)
lang_check = s.pivot_table(index="layer", columns=["position", "lang"], values=["p_target", "p_source"])
display(lang_check.iloc[::4].round(3))

### Per case: compare with the lens's failure-mode label
For each case: the layer where the fact-specific patch effect peaks, and whether any layer flips the answer to the new one. Then cross-tabulated against the label `failure_modes` gave from the lens profile in `run_rohan.ipynb`.

In [ ]:
from xling.evaluate import failure_modes
status = pd.read_csv(f"{RUN_DIR}/status.csv")
fm = failure_modes(status[status.model == "round2"])[["fact_id", "lang", "failure_mode"]]

per_case = (patch.sort_values("delta_specific", ascending=False)
                 .groupby(["position", "fact_id", "lang"], as_index=False).first()
                 [["position", "fact_id", "lang", "layer", "delta_specific", "D_base"]]
                 .rename(columns={"layer": "best_layer", "delta_specific": "best_delta_specific"}))
flips = patch.groupby(["position", "fact_id", "lang"]).flipped.any().rename("any_layer_flips").reset_index()
per_case = per_case.merge(flips, on=["position", "fact_id", "lang"]).merge(fm, on=["fact_id", "lang"], how="left")
per_case.to_csv(f"{RUN_DIR}/patching_per_case.csv", index=False)
display(per_case.groupby(["position", "lang"]).agg(n=("fact_id", "size"), median_best_layer=("best_layer", "median"),
                                                   share_flippable=("any_layer_flips", "mean")).round(2))
for pos in POSITIONS:
    d = per_case[per_case.position == pos]
    d = d.assign(best_layer_band=pd.cut(d.best_layer, [-1, 9, 19, 24, 28], labels=["0-9", "10-19", "20-24", "25-28"]))
    print(f"\n{pos} position: lens failure mode vs layer band of the best patch")
    display(pd.crosstab(d.failure_mode.fillna("no lens label"), d.best_layer_band))

## Notes
- **Few cases:** about 44 failures across three languages, one prompt each. Report the curves and per-language peaks as patterns, not significance tests.
- **Late-layer patches at the last position:** these mostly copy the English *output* into the target run (the last layer copies it exactly; check 3). The `p_source` column shows when that happens. The interesting range is the middle layers.
- **Other runs:** to patch your 30-epoch stand-in run instead, re-evaluate it first. The adapters now in `adapters/fictional` are last night's 100-epoch ones, and `results/fictional` came from the 30-epoch run. Then point `FACTS`, `ADAPTER` and `RUN_DIR` at it.